# Exploratory Analysis

This notebook contains the initial data exploration and model experiments for the bike taxi demand prediction project.

Place the dataset at `data/raw/ola.csv` before running the notebook.


In [ ]:
# Importing Libraries and Dataset
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sb 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import LabelEncoder, StandardScaler 
from sklearn.metrics import mean_absolute_error as mae
from sklearn.svm import SVR 
from xgboost import XGBRegressor 
from sklearn.linear_model import LinearRegression, Lasso, Ridge 
from sklearn.ensemble import RandomForestRegressor 
import holidays 
from datetime import datetime

import warnings 
warnings.filterwarnings('ignore')

In [ ]:
from pathlib import Path
DATA_PATH = Path("data/raw/ola.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Place the authorized dataset at {DATA_PATH.resolve()}")
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Mapping for 'season' column
season_mapping = {
    'spring': 1,
    'summer': 2,
    'fall': 3,
    'winter': 4
}

# Mapping for 'weather' column
weather_mapping = {
    'Clear, Few clouds, Partly cloudy, Partly cloudy': 1,
    'Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist': 2,
    'Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds': 3,
    'Heavy Rain + Ice Pallets + Thunderstorm + Mist, Snow + Fog': 4
}

# Apply mappings to existing columns, with case-insensitivity and stripping whitespace
df['season'] = df['season'].str.strip().str.lower().map(lambda x: season_mapping.get(x, x))
df['weather'] = df['weather'].str.strip().str.lower().map(lambda x: weather_mapping.get(x, x))

# Drop the 'season_mapped' and 'weather_mapped' columns if they exist
df.drop(['season_mapped', 'weather_mapped'], axis=1, inplace=True, errors='ignore')


In [ ]:
df.head()

In [ ]:
# Define mappings
weather_mapping = {
    'Clear': 1,
    'Cloudy': 2,
    'Thunderstorm': 3,
    'Heavy rain': 3,
    'Partly cloudy': 2,
    'Fog': 4,
    'Light Rain': 3,
    'Mist': 4,
    'Few clouds': 2
}

# Convert DataFrame values and dictionary keys to lowercase and remove leading/trailing spaces
df['weather'] = df['weather'].map(lambda value: value.lower().strip() if isinstance(value, str) else value)
weather_mapping = {key.lower().strip(): value for key, value in weather_mapping.items()}

# Replace the values in the DataFrame according to the modified mapping
df['weather'] = df['weather'].replace(weather_mapping)

# Display the DataFrame after mapping
print(df)


In [ ]:
df.info()


In [ ]:
df.describe().T


In [ ]:
parts = df["datetime"].str.split(" ", n=2, expand=True) 
df["date"] = parts[0] 
df["time"] = parts[1].str[:2].astype('int') 
df.head() 


In [ ]:
parts = df["date"].str.split("-", n=3, expand=True) 
df["year"] = parts[0].astype('int') 
df["month"] = parts[1].astype('int') 
df["day"] = parts[2].astype('int') 
df.head() 


In [ ]:
from datetime import datetime 
import calendar 


def weekend_or_weekday(year, month, day): 

	d = datetime(year, month, day) 
	if d.weekday() > 4: 
		return 0
	else: 
		return 1


df['weekday'] = df.apply(lambda x: 
						weekend_or_weekday(x['year'], 
											x['month'], 
											x['day']), 
						axis=1) 
df.head() 


In [ ]:
def am_or_pm(x): 
	if x > 11: 
		return 1
	else: 
		return 0


df['am_or_pm'] = df['time'].apply(am_or_pm) 
df.head() 


In [ ]:
from datetime import date 
import holidays 


def is_holiday(x): 

	india_holidays = holidays.country_holidays('IN') 

	if india_holidays.get(x): 
		return 1
	else: 
		return 0


df['holidays'] = df['date'].apply(is_holiday) 
df.head() 


In [ ]:
df = df.sort_values('datetime').reset_index(drop=True)
df.drop(['datetime', 'date'], 
		axis=1, 
		inplace=True) 


In [ ]:
df.isnull().sum()


In [ ]:
features = ['day', 'time', 'month'] 

plt.subplots(figsize=(15, 10)) 
for i, col in enumerate(features): 
	plt.subplot(2, 2, i + 1) 
	df.groupby(col).mean()['count'].plot() 
plt.show() 


In [ ]:
features = ['season', 'weather', 'holidays', 'am_or_pm', 'year', 'weekday']

plt.subplots(figsize=(20, 10))
for i, col in enumerate(features):
    plt.subplot(2, 3, i + 1)
    df.groupby(col).mean()['count'].plot.bar()
plt.show()


In [ ]:
features = ['temp (°C)', 'windspeed (m/s)'] 

plt.subplots(figsize=(15, 5)) 
for i, col in enumerate(features):

    plt.subplot(1, 2, i + 1) 
    sb.histplot(df[col], kde=True) 
plt.show()


In [ ]:
features = ['temp (°C)', 'windspeed (m/s)'] 

plt.subplots(figsize=(15, 5)) 
for i, col in enumerate(features):
   plt.subplot(1, 2, i + 1) 
   sb.boxplot(x=df[col]) 
plt.show()


In [ ]:
num_rows = df.shape[0] - df[df['windspeed (m/s)']<32].shape[0] 
print(f'Number of rows that will be lost if we remove outliers is equal to {num_rows}.')


In [ ]:
features = ['humidity (%)', 'casual_riders', 'registered_riders', 'count'] 

plt.subplots(figsize=(15, 10)) 
for i, col in enumerate(features): 
	plt.subplot(2, 2, i + 1) 
	sb.boxplot(x=df[col]) 
plt.show() 


In [ ]:
sb.heatmap(df.corr() > 0.8, 
		annot=True, 
		cbar=False) 
plt.show()


In [ ]:
# Check for the existence of columns before dropping
columns_to_drop = ['registered_riders', 'casual_riders', 'time']
existing_columns = [col for col in columns_to_drop if col in df.columns]

if existing_columns:
    df.drop(existing_columns, axis=1, inplace=True)

# Apply the filtering conditions
df = df[(df['windspeed (m/s)'] < 32) & (df['humidity (%)'] > 0)]


In [ ]:
# Keep the temporal order: earlier observations train the model; later observations validate it.
split_index = int(len(features) * 0.9)
X_train = features.iloc[:split_index].copy()
X_val = features.iloc[split_index:].copy()
Y_train = target[:split_index]
Y_val = target[split_index:]
X_train.shape, X_val.shape

In [ ]:
scaler = StandardScaler() 
X_train = scaler.fit_transform(X_train) 
X_val = scaler.transform(X_val) 


In [ ]:
models = [LinearRegression(), XGBRegressor(), Lasso(), RandomForestRegressor(), Ridge()]

for model in models:
    model.fit(X_train, Y_train)

    print(f'Model: {model.__class__.__name__}')
    
    train_preds = model.predict(X_train)
    train_error = mae(Y_train, train_preds)
    print(f'Training Error: {train_error:.3f}')

    val_preds = model.predict(X_val)
    val_error = mae(Y_val, val_preds)
    print(f'Validation Error: {val_error:.3f}')

    print()


In [ ]:
models = [LinearRegression(),XGBRegressor(),Lasso(),RandomForestRegressor(),Ridge()]

# Loop through each model
for model in models:
    # Fit the model on the training data
    model.fit(X_train, Y_train)
    
    # Print the model name
    print(f'{model.__class__.__name__} : ')
    
    # Predict on the training data
    train_preds = model.predict(X_train)
    # Calculate and print the training error
    print('Training Error : ', mae(Y_train, train_preds))
    
    # Predict on the validation data
    val_preds = model.predict(X_val)
    # Calculate and print the validation error
    print('Validation Error : ', mae(Y_val, val_preds))
    print()


In [ ]:
models = [LinearRegression(), XGBRegressor(), Lasso(), RandomForestRegressor(), Ridge()]

# Initialize lists to store results
train_errors = []
val_errors = []
predictions = []

for model in models:
    model.fit(X_train, Y_train)
    train_preds = model.predict(X_train)
    train_error = mae(Y_train, train_preds)
    val_preds = model.predict(X_val)
    val_error = mae(Y_val, val_preds)
    
    # Store results
    train_errors.append(train_error)
    val_errors.append(val_error)
    predictions.append(val_preds)
    
    # Print model results
    print(f'Model: {model.__class__.__name__}')
    print(f'Training Error: {train_error:.3f}')
    print(f'Validation Error: {val_error:.3f}')
    print()

# Display the final results
results = pd.DataFrame({
    'Model': [model.__class__.__name__ for model in models],
    'Training Error': train_errors,
    'Validation Error': val_errors
})

print(results)

# Display predictions from the best model (based on validation error)
best_model_index = val_errors.index(min(val_errors))
best_model = models[best_model_index]
best_predictions = predictions[best_model_index]

print(f'Best Model: {best_model.__class__.__name__}')
print(f'Predictions: {best_predictions[:20]}')  # Display the first 10 predictions as an example